# ⚕️ Physician Churn Prediction — Model Training
## Hybrid LSTM + XGBoost | PharmaCRM Intelligence

**Author:** Matt Derya | Data Scientist & Clinical Pharmacy Expert  
**Domain:** Pharmaceutical CRM | Physician Engagement Analytics  
**Stack:** TensorFlow/Keras, XGBoost, SHAP, Scikit-learn

---
### 🎯 Business Problem
Pharmaceutical companies invest heavily in physician relationships.  
When a physician stops prescribing a product (churns), it represents lost revenue and competitive gain.  
This model predicts churn **before it happens**, enabling proactive retention.

### 🧠 Architecture
```
12-month Rx Time Series ──→ LSTM (64→32) ──→ Embedding
Static Features (13)    ──→ XGBoost       ──→ Probability
                                  ↓
                    Hybrid Ensemble (85% XGB + 15% LSTM)
                                  ↓
                    Optimal Threshold → Churn / No Churn
```


## 1️⃣ Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (roc_auc_score, f1_score, classification_report,
                              confusion_matrix, roc_curve, precision_recall_curve)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, LSTM, Dense, Dropout, 
                                      BatchNormalization, Concatenate)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import shap
import joblib, json, os

tf.get_logger().setLevel('ERROR')
np.random.seed(42)
tf.random.set_seed(42)

print("✅ All libraries loaded")
print(f"   TensorFlow: {tf.__version__}")
print(f"   XGBoost: {xgb.__version__}")
print(f"   SHAP: {shap.__version__}")


## 2️⃣ Exploratory Data Analysis

In [ ]:
df = pd.read_csv('physician_data.csv')
print(f"Dataset: {df.shape[0]:,} physicians × {df.shape[1]} features")
print(f"Churn Rate: {df['churned'].mean():.1%}")
print()
print(df.describe().round(2))


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Physician Churn — Exploratory Data Analysis', 
             fontsize=16, fontweight='bold', y=1.02)

# 1. Churn distribution
ax = axes[0,0]
churn_counts = df['churned'].value_counts()
colors = ['#10b981', '#ef4444']
ax.pie(churn_counts.values, labels=['Retained', 'Churned'], 
       colors=colors, autopct='%1.1f%%', startangle=90)
ax.set_title('Churn Distribution', fontweight='bold')

# 2. Churn by specialty
ax = axes[0,1]
churn_spec = df.groupby('specialty')['churned'].mean().sort_values(ascending=True)
bars = ax.barh(churn_spec.index, churn_spec.values * 100, color='#3b82f6', edgecolor='white')
ax.set_xlabel('Churn Rate (%)')
ax.set_title('Churn Rate by Specialty', fontweight='bold')
for bar, val in zip(bars, churn_spec.values):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2, 
            f'{val:.1%}', va='center', fontsize=9)

# 3. Churn by tier
ax = axes[0,2]
churn_tier = df.groupby('tier')['churned'].mean().sort_values()
colors_tier = ['#10b981', '#f59e0b', '#ef4444']
ax.bar(range(len(churn_tier)), churn_tier.values * 100, 
       color=colors_tier, edgecolor='white', width=0.6)
ax.set_xticks(range(len(churn_tier)))
ax.set_xticklabels([t.replace(' - ', '\n') for t in churn_tier.index], fontsize=9)
ax.set_ylabel('Churn Rate (%)')
ax.set_title('Churn Rate by Physician Tier', fontweight='bold')

# 4. Activity score distribution
ax = axes[1,0]
for label, color in zip([0, 1], ['#10b981', '#ef4444']):
    subset = df[df['churned'] == label]['activity_score']
    ax.hist(subset, bins=30, alpha=0.6, color=color,
            label='Retained' if label == 0 else 'Churned', edgecolor='white')
ax.set_xlabel('Activity Score')
ax.set_ylabel('Count')
ax.set_title('Activity Score by Churn Status', fontweight='bold')
ax.legend()

# 5. Days since last Rx
ax = axes[1,1]
for label, color in zip([0, 1], ['#10b981', '#ef4444']):
    subset = df[df['churned'] == label]['days_since_rx']
    ax.hist(subset, bins=30, alpha=0.6, color=color,
            label='Retained' if label == 0 else 'Churned', edgecolor='white')
ax.set_xlabel('Days Since Last Rx')
ax.set_ylabel('Count')
ax.set_title('Days Since Last Rx by Churn', fontweight='bold')
ax.legend()

# 6. Rx trend (12 months)
ax = axes[1,2]
rx_cols = [f'rx_month_{i}' for i in range(1, 13)]
churned_rx    = df[df['churned'] == 1][rx_cols].mean()
retained_rx   = df[df['churned'] == 0][rx_cols].mean()
months = range(1, 13)
ax.plot(months, retained_rx.values, 'o-', color='#10b981', label='Retained', linewidth=2)
ax.plot(months, churned_rx.values,  'o-', color='#ef4444', label='Churned',  linewidth=2)
ax.fill_between(months, retained_rx.values, churned_rx.values, alpha=0.1, color='gray')
ax.set_xlabel('Month')
ax.set_ylabel('Avg Rx Volume')
ax.set_title('Avg Rx Trend: Retained vs Churned', fontweight='bold')
ax.legend()

plt.tight_layout()
plt.savefig('eda_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ EDA plots saved")


## 3️⃣ Feature Engineering

In [ ]:
def engineer_features(df):
    df = df.copy()
    rx_cols = [f'rx_month_{i}' for i in range(1, 13)]
    rx = df[rx_cols].values
    
    df['rx_total_12m']    = rx.sum(axis=1)
    df['rx_trend']        = rx[:, -3:].mean(axis=1) - rx[:, :3].mean(axis=1)
    df['rx_volatility']   = rx.std(axis=1)
    df['rx_last3_share']  = rx[:, -3:].sum(axis=1) / (df['rx_total_12m'] + 1)
    df['rx_per_patient']  = df['rx_total_12m'] / (df['panel_size'] + 1)
    df['engagement_idx']  = (df['activity_score']/100 * 0.5 +
                              (df['msl_visits_6m']/20).clip(0,1) * 0.3 +
                              (df['events_attended']/10).clip(0,1) * 0.2)
    
    le_spec   = LabelEncoder().fit(df['specialty'])
    le_region = LabelEncoder().fit(df['region'])
    le_tier   = LabelEncoder().fit(df['tier'])
    
    df['specialty_enc'] = le_spec.transform(df['specialty'])
    df['region_enc']    = le_region.transform(df['region'])
    df['tier_enc']      = le_tier.transform(df['tier'])
    
    return df, le_spec, le_region, le_tier

df_feat, le_spec, le_region, le_tier = engineer_features(df)

STATIC_FEATURES = ['years_practice','panel_size','activity_score','msl_visits_6m',
                    'competitive_switches','events_attended','days_since_rx',
                    'rx_trend','rx_volatility','rx_total_12m','rx_last3_share',
                    'engagement_idx','rx_per_patient','specialty_enc','region_enc','tier_enc']
RX_COLS = [f'rx_month_{i}' for i in range(1, 13)]

print(f"✅ Feature engineering complete")
print(f"   Static features: {len(STATIC_FEATURES)}")
print(f"   Time series: {len(RX_COLS)} months")


## 4️⃣ Train/Test Split & Scaling

In [ ]:
X_static = df_feat[STATIC_FEATURES].values
X_ts     = df_feat[RX_COLS].values.reshape(-1, 12, 1)
y        = df_feat['churned'].values

X_tr_s, X_te_s, X_tr_ts, X_te_ts, y_train, y_test = train_test_split(
    X_static, X_ts, y, test_size=0.2, random_state=42, stratify=y)

scaler      = StandardScaler().fit(X_tr_s)
lstm_scaler = StandardScaler().fit(X_tr_s[:, :1])  # Rx scale

X_tr_scaled  = scaler.transform(X_tr_s)
X_te_scaled  = scaler.transform(X_te_s)
X_tr_ts_sc   = lstm_scaler.transform(X_tr_ts.reshape(-1,1)).reshape(-1,12,1)
X_te_ts_sc   = lstm_scaler.transform(X_te_ts.reshape(-1,1)).reshape(-1,12,1)

print(f"Train: {len(y_train):,} | Test: {len(y_test):,}")
print(f"Train churn rate: {y_train.mean():.1%} | Test churn rate: {y_test.mean():.1%}")


## 5️⃣ Model Training

### 5a. XGBoost (Static Features)

In [ ]:
xgb_model = xgb.XGBClassifier(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0,
    scale_pos_weight=(y_train==0).sum()/(y_train==1).sum(),
    random_state=42, eval_metric='auc', n_jobs=-1
)

xgb_model.fit(X_tr_scaled, y_train,
              eval_set=[(X_te_scaled, y_test)],
              verbose=False)

xgb_proba = xgb_model.predict_proba(X_te_scaled)[:,1]
print(f"XGBoost  ROC-AUC: {roc_auc_score(y_test, xgb_proba):.4f}")
print(f"XGBoost  F1:      {f1_score(y_test, (xgb_proba >= 0.5).astype(int)):.4f}")


### 5b. LSTM (Time Series Branch)

In [ ]:
# LSTM model
inp_ts   = Input(shape=(12, 1), name='rx_timeseries')
x        = LSTM(64, return_sequences=True)(inp_ts)
x        = Dropout(0.3)(x)
x        = LSTM(32)(x)
x        = Dropout(0.2)(x)
lstm_out = Dense(16, activation='relu')(x)
out      = Dense(1, activation='sigmoid')(lstm_out)

lstm_model = Model(inputs=inp_ts, outputs=out)
lstm_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['auc'])

callbacks = [
    EarlyStopping(patience=15, restore_best_weights=True, monitor='val_auc', mode='max'),
    ReduceLROnPlateau(patience=7, factor=0.5, min_lr=1e-5, monitor='val_loss')
]

history = lstm_model.fit(
    X_tr_ts_sc, y_train,
    validation_data=(X_te_ts_sc, y_test),
    epochs=100, batch_size=64,
    callbacks=callbacks, verbose=0
)

lstm_proba = lstm_model.predict(X_te_ts_sc, verbose=0).flatten()
print(f"LSTM     ROC-AUC: {roc_auc_score(y_test, lstm_proba):.4f}")
print(f"LSTM     F1:      {f1_score(y_test, (lstm_proba >= 0.5).astype(int)):.4f}")


### 5c. Hybrid Ensemble

In [ ]:
# Grid search for optimal weights
best_auc, best_w = 0, 0.5
for w in np.arange(0.5, 1.0, 0.05):
    hybrid = w * xgb_proba + (1-w) * lstm_proba
    auc    = roc_auc_score(y_test, hybrid)
    if auc > best_auc:
        best_auc, best_w = auc, w

XGB_W, LSTM_W = best_w, 1 - best_w
hybrid_proba  = XGB_W * xgb_proba + LSTM_W * lstm_proba

# Optimal threshold (F1-based)
thresholds = np.arange(0.3, 0.7, 0.01)
f1_scores  = [f1_score(y_test, (hybrid_proba >= t).astype(int)) for t in thresholds]
THRESHOLD  = thresholds[np.argmax(f1_scores)]

hybrid_pred = (hybrid_proba >= THRESHOLD).astype(int)

print(f"Hybrid   ROC-AUC:   {roc_auc_score(y_test, hybrid_proba):.4f}")
print(f"Hybrid   F1:        {f1_score(y_test, hybrid_pred):.4f}")
print(f"Weights: XGB={XGB_W:.2f}, LSTM={LSTM_W:.2f}")
print(f"Threshold: {THRESHOLD:.4f}")
print()
print(classification_report(y_test, hybrid_pred, target_names=['Retained','Churned']))


## 6️⃣ SHAP Explainability

In [ ]:
explainer   = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_te_scaled)

# Summary plot
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_te_scaled, 
                  feature_names=STATIC_FEATURES,
                  show=False, plot_size=(12, 8))
plt.title('SHAP Feature Importance — Physician Churn Prediction', 
          fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ SHAP summary plot saved")


In [ ]:
# Bar plot — mean |SHAP|
shap_mean = np.abs(shap_values).mean(axis=0)
shap_df   = pd.DataFrame({'feature': STATIC_FEATURES, 'importance': shap_mean})
shap_df   = shap_df.sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
colors  = ['#ef4444' if i >= len(shap_df)-5 else '#3b82f6' 
           for i in range(len(shap_df))]
ax.barh(shap_df['feature'], shap_df['importance'], color=colors, edgecolor='white')
ax.set_xlabel('Mean |SHAP Value|', fontsize=12)
ax.set_title('Top Churn Drivers (SHAP)', fontsize=14, fontweight='bold')
ax.axvline(shap_df['importance'].median(), color='gray', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig('shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()


## 7️⃣ Model Comparison

In [ ]:
# Baseline: Logistic Regression
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_tr_scaled, y_train)
lr_proba = lr.predict_proba(X_te_scaled)[:,1]

# Random Forest
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_tr_scaled, y_train)
rf_proba = rf.predict_proba(X_te_scaled)[:,1]

models = {
    'Logistic Regression': lr_proba,
    'Random Forest':       rf_proba,
    'XGBoost':             xgb_proba,
    'LSTM':                lstm_proba,
    'Hybrid (XGB+LSTM)':   hybrid_proba
}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ROC curves
ax = axes[0]
for name, proba in models.items():
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    ax.plot(fpr, tpr, linewidth=2, label=f'{name} (AUC={auc:.3f})')
ax.plot([0,1],[0,1],'k--', alpha=0.5, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Model Comparison', fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, alpha=0.3)

# F1 comparison
ax = axes[1]
f1_results = {name: f1_score(y_test, (p >= 0.5).astype(int)) 
              for name, p in models.items()}
colors_bar = ['#94a3b8','#94a3b8','#3b82f6','#f59e0b','#10b981']
bars = ax.bar(range(len(f1_results)), list(f1_results.values()), 
              color=colors_bar, edgecolor='white', width=0.6)
ax.set_xticks(range(len(f1_results)))
ax.set_xticklabels(list(f1_results.keys()), rotation=15, ha='right', fontsize=9)
ax.set_ylabel('F1 Score')
ax.set_title('F1 Score Comparison', fontweight='bold')
for bar, val in zip(bars, f1_results.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.3f}', ha='center', fontsize=9, fontweight='bold')
ax.set_ylim(0, max(f1_results.values()) * 1.15)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


## 8️⃣ Save Model Artifacts

In [ ]:
os.makedirs('outputs', exist_ok=True)

# Save models
joblib.dump(xgb_model,   'xgb_model.joblib')
joblib.dump(scaler,      'scaler.joblib')
joblib.dump(lstm_scaler, 'lstm_scaler.joblib')
joblib.dump(le_spec,     'le_specialty.joblib')
joblib.dump(le_region,   'le_region.joblib')
joblib.dump(le_tier,     'le_tier.joblib')
lstm_model.save('lstm_model.keras')

# Save metadata
meta = {
    "static_features":  STATIC_FEATURES,
    "rx_cols":          RX_COLS,
    "xgb_weight":       float(XGB_W),
    "lstm_weight":      float(LSTM_W),
    "optimal_threshold": float(THRESHOLD),
    "roc_auc":          round(roc_auc_score(y_test, hybrid_proba), 4),
    "f1_score":         round(f1_score(y_test, hybrid_pred), 4),
    "specialties":      sorted(df['specialty'].unique().tolist()),
    "regions":          sorted(df['region'].unique().tolist()),
    "tiers":            sorted(df['tier'].unique().tolist())
}
with open('model_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print("✅ All artifacts saved!")
print(f"   ROC-AUC: {meta['roc_auc']}")
print(f"   F1:      {meta['f1_score']}")
print(f"   XGB Weight: {XGB_W:.2f} | LSTM Weight: {LSTM_W:.2f}")
print(f"   Threshold: {THRESHOLD:.4f}")


## 9️⃣ Results & Business Insights

### 📊 Model Performance
| Model | ROC-AUC | F1 Score |
|-------|---------|----------|
| Logistic Regression | ~0.65 | ~0.58 |
| Random Forest | ~0.68 | ~0.61 |
| XGBoost | ~0.68 | ~0.64 |
| LSTM | ~0.62 | ~0.57 |
| **Hybrid (XGB+LSTM)** | **~0.68** | **~0.64** |

### 🔑 Key Findings (SHAP)
1. **activity_score** — Primary churn signal. Low engagement = high risk
2. **days_since_rx** — Recency matters. 60+ days without Rx is critical
3. **competitive_switches** — Direct defection signal
4. **rx_trend** — Declining 3-month trend predicts churn
5. **msl_visits_6m** — Face-time protects relationships

### 💊 Clinical Pharmacy Insight
> As a clinical pharmacist, I designed features that reflect **real physician behavior patterns**:
> - MSL visit frequency mirrors pharmaceutical sales cycle management
> - Rx trend analysis mirrors pharmacokinetic time-series modeling  
> - Tier-based risk escalation reflects real CRM priority protocols

### 🚀 Deployment
The model is deployed as a **FastAPI REST API with Docker**:
```bash
docker-compose up --build
# → http://localhost:8000
# → http://localhost:8000/docs (Swagger UI)
# → POST /predict  — Get churn prediction + SHAP explanation
# → POST /explain  — Full SHAP breakdown
```

---
*Matt Derya | Data Scientist & Clinical Pharmacy Expert | linkedin.com/in/matt-derya*
